# 11 - Noise, Decoherence, and Density Matrices

Real quantum hardware does not evolve unitarily forever. Every qubit is an
**open quantum system** -- it interacts with its environment, and those
interactions introduce errors. Understanding noise is essential for building
practical quantum algorithms, because a circuit that works perfectly in
simulation may produce garbage on a real device.

In this notebook we will:

1. Move from statevectors to **density matrices**, the natural language for
   mixed (noisy) quantum states.
2. Explore **noise channels** -- the mathematical description of how errors
   affect qubits -- via Kraus operators.
3. Quantify noise with **purity** and **fidelity**.
4. Survey the standard channels: depolarizing, amplitude damping, phase
   damping, bit flip, and phase flip.
5. Model realistic decoherence with **T1/T2 thermal relaxation**.
6. Build **custom noise channels** from scratch.

### Misconception: Quantum error correction is just classical error correction

Classical error correction works by copying bits. Quantum error correction
cannot do this -- the **no-cloning theorem** forbids copying an unknown
quantum state. Instead, quantum EC encodes logical qubits into entangled
states of many physical qubits, using syndrome measurements that extract
error information without disturbing the encoded data. This is a
fundamentally different (and harder) problem.

In [ ]:
import (
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/sim/densitymatrix"
	"github.com/splch/goqu/sim/noise"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## From Statevectors to Density Matrices

So far we have represented quantum states as **statevectors** $|\psi\rangle$.
This works perfectly for isolated systems undergoing unitary evolution. But
when a qubit interacts with its environment, the system's state can no longer
be described by a single statevector -- it becomes a statistical mixture of
pure states.

The **density matrix** $\rho$ handles both cases:

- **Pure state:** $\rho = |\psi\rangle\langle\psi|$, with $\text{Tr}(\rho^2) = 1$
- **Mixed state:** $\rho = \sum_i p_i |\psi_i\rangle\langle\psi_i|$, with $\text{Tr}(\rho^2) < 1$

The quantity $\text{Tr}(\rho^2)$ is called the **purity**. It equals 1 for
pure states and reaches a minimum of $1/d$ (where $d$ is the Hilbert space
dimension) for the maximally mixed state $\rho = I/d$.

**Fidelity** $F = \langle\psi|\rho|\psi\rangle$ measures how close a noisy
state $\rho$ is to an ideal pure state $|\psi\rangle$. $F = 1$ means perfect,
$F = 1/d$ means no better than random.

Let's create a Bell state using the density matrix simulator and verify it
is pure.

In [ ]:
%%
// Build a Bell state: H(0), CNOT(0,1)
cBell, err := builder.New("bell", 2).H(0).CNOT(0, 1).Build()
if err != nil {
	panic(err)
}

fmt.Println("Bell circuit:")
fmt.Println(draw.String(cBell))

// Create a density matrix simulator (no noise)
dm := densitymatrix.New(2)
if err := dm.Evolve(cBell); err != nil {
	panic(err)
}

// Inspect the density matrix
rho := dm.DensityMatrix()
dim := 4 // 2^2
fmt.Println("Density matrix (non-zero elements):")
for r := 0; r < dim; r++ {
	for c := 0; c < dim; c++ {
		v := rho[r*dim+c]
		if real(v)*real(v)+imag(v)*imag(v) > 1e-10 {
			fmt.Printf("  rho[|%02b>,|%02b>] = %+.4f\n", r, c, real(v))
		}
	}
}

fmt.Printf("\nPurity: %.6f\n", dm.Purity())
fmt.Printf("Is pure (purity ~ 1.0): %v\n", math.Abs(dm.Purity()-1.0) < 1e-10)

// Get the ideal statevector for fidelity comparisons later
svSim := statevector.New(2)
if err := svSim.Evolve(cBell); err != nil {
	panic(err)
}
idealSV := svSim.StateVector()
fmt.Printf("Fidelity vs ideal: %.6f\n", dm.Fidelity(idealSV))

## Noise Channels

A **noise channel** describes how the environment corrupts a quantum state.
Mathematically, it is a completely positive, trace-preserving (CPTP) map
expressed via **Kraus operators** $\{E_k\}$:

$$\rho \mapsto \sum_k E_k \, \rho \, E_k^\dagger$$

The trace-preservation condition $\sum_k E_k^\dagger E_k = I$ guarantees
probabilities still sum to 1 after the noise is applied.

In Goqu, we attach noise channels to gates via a `NoiseModel`. After each
gate is applied, its associated noise channel is automatically applied to
the density matrix. Let's see how depolarizing noise degrades our Bell state.

In [ ]:
%%
// Create a noise model with depolarizing noise on H and CNOT gates
nm := noise.New()
nm.AddGateError("H", noise.Depolarizing1Q(0.01))
nm.AddGateError("CNOT", noise.Depolarizing2Q(0.02))

// Simulate the Bell circuit with noise
dmNoisy := densitymatrix.New(2)
dmNoisy.WithNoise(nm)
if err := dmNoisy.Evolve(cBell); err != nil {
	panic(err)
}

// Compare ideal vs noisy
fmt.Println("=== Ideal (noiseless) ===")
fmt.Printf("  Purity:   %.6f\n", dm.Purity())
fmt.Printf("  Fidelity: %.6f\n", dm.Fidelity(idealSV))

fmt.Println("\n=== Noisy (1%% depolarizing on H, 2%% on CNOT) ===")
fmt.Printf("  Purity:   %.6f\n", dmNoisy.Purity())
fmt.Printf("  Fidelity: %.6f\n", dmNoisy.Fidelity(idealSV))

fmt.Println("\nNoise reduces both purity and fidelity.")
fmt.Println("Purity < 1 means the state is mixed (no longer a pure quantum state).")
fmt.Println("Fidelity < 1 means the state has drifted from the ideal target.")

## Types of Noise Channels

Different physical mechanisms produce different kinds of noise:

| Channel | Physical origin | Effect |
|:---|:---|:---|
| **Depolarizing** | Isotropic noise; random Pauli errors (X, Y, Z) with equal probability | Shrinks the Bloch vector uniformly toward the origin |
| **Amplitude damping** | Energy relaxation (T1 decay); spontaneous emission | Drives $|1\rangle \to |0\rangle$; the qubit loses energy |
| **Phase damping** | Pure dephasing (T2 without T1); frequency fluctuations | Destroys off-diagonal coherence; no population change |
| **Bit flip** | Random X error | Flips $|0\rangle \leftrightarrow |1\rangle$ with probability $p$ |
| **Phase flip** | Random Z error | Flips phase of $|1\rangle$ with probability $p$ |

Let's apply each channel (with the same error parameter $p = 0.1$) to a
simple $H|0\rangle = |+\rangle$ circuit and compare their effects on purity
and fidelity.

In [ ]:
%%
// Single-qubit H circuit: |0> -> |+>
cH, err := builder.New("h", 1).H(0).Build()
if err != nil {
	panic(err)
}

// Ideal statevector for fidelity reference
svIdeal := statevector.New(1)
if err := svIdeal.Evolve(cH); err != nil {
	panic(err)
}
idealPlus := svIdeal.StateVector()

// Define channels to compare
type channelEntry struct {
	name string
	ch   noise.Channel
}
channels := []channelEntry{
	{"Depolarizing(0.1)", noise.Depolarizing1Q(0.1)},
	{"AmplitudeDamping(0.1)", noise.AmplitudeDamping(0.1)},
	{"PhaseDamping(0.1)", noise.PhaseDamping(0.1)},
	{"BitFlip(0.1)", noise.BitFlip(0.1)},
	{"PhaseFlip(0.1)", noise.PhaseFlip(0.1)},
}

fmt.Printf("%-28s  %10s  %10s\n", "Channel", "Purity", "Fidelity")
fmt.Println("------------------------------------------------------")

for _, entry := range channels {
	nm := noise.New()
	nm.AddGateError("H", entry.ch)

	sim := densitymatrix.New(1)
	sim.WithNoise(nm)
	if err := sim.Evolve(cH); err != nil {
		panic(err)
	}

	fmt.Printf("%-28s  %10.6f  %10.6f\n", entry.name, sim.Purity(), sim.Fidelity(idealPlus))
}

fmt.Println("\nNote: Different channels affect the state differently.")
fmt.Println("Phase flip hurts |+> the most because |+> is a superposition in the Z basis.")
fmt.Println("Amplitude damping pushes toward |0>, which partially overlaps |+>.")

## Increasing Noise Strength

How does fidelity degrade as we increase the depolarizing noise parameter $p$?
For a single H gate with depolarizing noise, as $p$ goes from 0 to the
maximum mixing value, the output state smoothly transitions from the ideal
$|+\rangle$ to the maximally mixed state $I/2$.

Let's sweep $p$ from 0 to 0.5 and observe the fidelity curve.

In [ ]:
%%
// Sweep depolarizing noise from p=0 to p=0.5
fmt.Printf("%6s  %10s  %10s\n", "p", "Purity", "Fidelity")
fmt.Println("-------------------------------")

for i := 0; i <= 10; i++ {
	p := float64(i) * 0.05

	nm := noise.New()
	nm.AddGateError("H", noise.Depolarizing1Q(p))

	sim := densitymatrix.New(1)
	sim.WithNoise(nm)
	if err := sim.Evolve(cH); err != nil {
		panic(err)
	}

	fmt.Printf("%6.2f  %10.6f  %10.6f\n", p, sim.Purity(), sim.Fidelity(idealPlus))
}

fmt.Println("\nAs p increases, both purity and fidelity decrease.")
fmt.Println("At p=0.75 (not shown), the output would be completely mixed: purity = 0.5, fidelity = 0.5.")

## Density Matrix Visualization

The **state city plot** is an isometric 3D bar chart of the density matrix
elements. Each bar shows the real or imaginary part of $\rho_{ij}$. For a
pure Bell state, the density matrix has four non-zero entries (the corners of
the $|00\rangle$/$|11\rangle$ subspace). Noise smears this structure, adding
weight to other elements and reducing the height of the ideal peaks.

Let's visualize the ideal Bell state and a noisy version side by side.

In [ ]:
%%
// Ideal Bell state density matrix
dmIdeal := densitymatrix.New(2)
if err := dmIdeal.Evolve(cBell); err != nil {
	panic(err)
}
rhoIdeal := dmIdeal.DensityMatrix()

// Noisy Bell state (5% depolarizing on all gates)
nmViz := noise.New()
nmViz.AddGateError("H", noise.Depolarizing1Q(0.05))
nmViz.AddGateError("CNOT", noise.Depolarizing2Q(0.10))

dmNoisyViz := densitymatrix.New(2)
dmNoisyViz.WithNoise(nmViz)
if err := dmNoisyViz.Evolve(cBell); err != nil {
	panic(err)
}
rhoNoisy := dmNoisyViz.DensityMatrix()

// Display state city plots
svgIdeal := viz.StateCity(rhoIdeal, 4, viz.WithTitle("Ideal Bell State"))
gonbui.DisplayHTML(svgIdeal)

svgNoisy := viz.StateCity(rhoNoisy, 4, viz.WithTitle("Noisy Bell State (5%/10% depolarizing)"))
gonbui.DisplayHTML(svgNoisy)

fmt.Printf("Ideal  -- Purity: %.4f, Fidelity: %.4f\n", dmIdeal.Purity(), dmIdeal.Fidelity(idealSV))
fmt.Printf("Noisy  -- Purity: %.4f, Fidelity: %.4f\n", dmNoisyViz.Purity(), dmNoisyViz.Fidelity(idealSV))

## Per-Qubit Noise and Readout Errors

Real devices do not have uniform noise across all qubits. Some qubits are
noisier than others due to fabrication variations. Goqu's noise model
supports **qubit-specific gate errors** and **readout errors**.

The resolution order for noise lookup is:
1. Gate name + specific qubits (most specific)
2. Gate name (any qubits)
3. Qubit count default (least specific)

**Readout errors** model classical measurement mistakes: $P(1|0)$ is the
probability of reading 1 when the true state is 0, and $P(0|1)$ is the
probability of reading 0 when the true state is 1.

In [ ]:
%%
// Per-qubit noise: qubit 0 is noisier than qubit 1
nmPerQubit := noise.New()

// Default H noise: 1%
nmPerQubit.AddGateError("H", noise.Depolarizing1Q(0.01))

// But qubit 0 has 5% noise on H (overrides the 1% default for this qubit)
nmPerQubit.AddGateQubitError("H", []int{0}, noise.Depolarizing1Q(0.05))

// CNOT noise
nmPerQubit.AddGateError("CNOT", noise.Depolarizing2Q(0.02))

// Readout errors: qubit 0 has 2% P(1|0) and 3% P(0|1)
nmPerQubit.AddReadoutError(0, noise.NewReadoutError(0.02, 0.03))
nmPerQubit.AddReadoutError(1, noise.NewReadoutError(0.01, 0.01))

// Evolve with per-qubit noise
dmPerQubit := densitymatrix.New(2)
dmPerQubit.WithNoise(nmPerQubit)
if err := dmPerQubit.Evolve(cBell); err != nil {
	panic(err)
}

fmt.Println("Per-qubit noise model:")
fmt.Println("  H on qubit 0: 5% depolarizing (qubit-specific override)")
fmt.Println("  H on qubit 1: 1% depolarizing (gate-level default)")
fmt.Println("  CNOT:         2% depolarizing")
fmt.Println("  Readout q0:   P(1|0)=0.02, P(0|1)=0.03")
fmt.Println("  Readout q1:   P(1|0)=0.01, P(0|1)=0.01")
fmt.Printf("\nPurity:   %.6f\n", dmPerQubit.Purity())
fmt.Printf("Fidelity: %.6f\n", dmPerQubit.Fidelity(idealSV))

// Sample measurements with readout error
cBellMeas, _ := builder.New("bell-meas", 2).H(0).CNOT(0, 1).MeasureAll().Build()
counts, err := dmPerQubit.Run(cBellMeas, 1000)
if err != nil {
	panic(err)
}

fmt.Println("\nMeasurement counts (1000 shots, includes readout error):")
for outcome, count := range counts {
	fmt.Printf("  |%s> : %d\n", outcome, count)
}
fmt.Println("\nNote: |01> and |10> appear due to both gate noise and readout errors.")

## Custom Noise Channels

Sometimes the built-in channels are not enough. You may want to model a
specific error process from a hardware characterization experiment. Goqu
lets you define a **custom channel** from arbitrary Kraus operators, as long
as they satisfy the trace-preservation condition $\sum_k E_k^\dagger E_k = I$.

Let's build a custom asymmetric depolarizing channel where X errors are more
likely than Y or Z errors.

In [ ]:
%%
// Custom asymmetric depolarizing channel:
// P(X error) = 0.06, P(Y error) = 0.02, P(Z error) = 0.02
// P(no error) = 1 - 0.06 - 0.02 - 0.02 = 0.90
pI := math.Sqrt(0.90)
pX := math.Sqrt(0.06)
pY := math.Sqrt(0.02)
pZ := math.Sqrt(0.02)

customCh := noise.MustCustom("asymmetric_depol", 1, [][]complex128{
	// sqrt(0.90) * I
	{complex(pI, 0), 0, 0, complex(pI, 0)},
	// sqrt(0.06) * X
	{0, complex(pX, 0), complex(pX, 0), 0},
	// sqrt(0.02) * Y
	{0, complex(0, -pY), complex(0, pY), 0},
	// sqrt(0.02) * Z
	{complex(pZ, 0), 0, 0, complex(-pZ, 0)},
})

// Apply to H gate
nmCustom := noise.New()
nmCustom.AddGateError("H", customCh)

dmCustom := densitymatrix.New(1)
dmCustom.WithNoise(nmCustom)
if err := dmCustom.Evolve(cH); err != nil {
	panic(err)
}

fmt.Println("Custom asymmetric depolarizing channel:")
fmt.Println("  P(no error) = 0.90, P(X) = 0.06, P(Y) = 0.02, P(Z) = 0.02")
fmt.Printf("\nPurity:   %.6f\n", dmCustom.Purity())
fmt.Printf("Fidelity: %.6f\n", dmCustom.Fidelity(idealPlus))

// Compare with symmetric depolarizing at same total error rate (p=0.10)
nmSym := noise.New()
nmSym.AddGateError("H", noise.Depolarizing1Q(0.10))
dmSym := densitymatrix.New(1)
dmSym.WithNoise(nmSym)
if err := dmSym.Evolve(cH); err != nil {
	panic(err)
}

fmt.Printf("\nComparison with symmetric Depolarizing1Q(0.10):\n")
fmt.Printf("  Symmetric purity:   %.6f\n", dmSym.Purity())
fmt.Printf("  Symmetric fidelity: %.6f\n", dmSym.Fidelity(idealPlus))
fmt.Println("\nDifferent error distributions produce different fidelity even at the same total error rate.")

## Thermal Relaxation (T1/T2)

The most physically realistic single-qubit noise model combines two time
scales:

- **T1** (energy relaxation time): how quickly the qubit decays from $|1\rangle$
  to $|0\rangle$ via spontaneous emission. This is amplitude damping.
- **T2** (dephasing time): how quickly quantum coherence (off-diagonal elements
  of $\rho$) decays. This includes both T1 effects and pure dephasing.

The physical constraint is $T_2 \leq 2 T_1$ (you cannot dephase slower than
the T1 limit). When $T_2 = 2 T_1$, all dephasing comes from energy relaxation
alone. When $T_2 < 2 T_1$, there is additional pure dephasing.

The `ThermalRelaxation(t1, t2, time)` channel composes amplitude damping
with residual phase damping to match the target T1 and T2 coherence times
for a gate of the given duration.

In [ ]:
%%
// Model thermal relaxation with T1=100ns, T2=60ns, gate time=20ns
t1 := 100.0 // T1 in nanoseconds
t2 := 60.0  // T2 in nanoseconds (T2 <= 2*T1)
gateTime := 20.0 // gate duration in nanoseconds

trCh := noise.ThermalRelaxation(t1, t2, gateTime)

nmTR := noise.New()
nmTR.AddGateError("H", trCh)

dmTR := densitymatrix.New(1)
dmTR.WithNoise(nmTR)
if err := dmTR.Evolve(cH); err != nil {
	panic(err)
}

fmt.Printf("Thermal Relaxation: T1=%.0fns, T2=%.0fns, gate=%.0fns\n", t1, t2, gateTime)
fmt.Printf("  Purity:   %.6f\n", dmTR.Purity())
fmt.Printf("  Fidelity: %.6f\n", dmTR.Fidelity(idealPlus))

// Sweep different gate durations to see decoherence over time
fmt.Printf("\n%10s  %10s  %10s\n", "Gate (ns)", "Purity", "Fidelity")
fmt.Println("-----------------------------------")

for _, gt := range []float64{0, 5, 10, 20, 40, 60, 80, 100} {
	ch := noise.ThermalRelaxation(t1, t2, gt)
	nm := noise.New()
	nm.AddGateError("H", ch)

	sim := densitymatrix.New(1)
	sim.WithNoise(nm)
	if err := sim.Evolve(cH); err != nil {
		panic(err)
	}

	fmt.Printf("%10.0f  %10.6f  %10.6f\n", gt, sim.Purity(), sim.Fidelity(idealPlus))
}

fmt.Println("\nLonger gate times mean more decoherence.")
fmt.Println("This is why fast gates are essential for quantum computing.")

## Predict, Then Verify

**Question:** What happens to the purity of a single-qubit $|+\rangle$ state
as depolarizing noise strength increases from $p = 0$ to $p = 0.75$?

**Prediction:** At $p = 0$, the state is pure: purity = 1. Depolarizing noise
mixes in the maximally mixed state. At $p = 0.75$ (maximum mixing for
single-qubit depolarizing), the output should be $I/2$ regardless of input,
giving purity = $\text{Tr}((I/2)^2) = 1/2$. Between these extremes, purity
should decrease smoothly.

In [ ]:
%%
// Verify: purity vs depolarizing noise from p=0 to p=0.75
fmt.Printf("%6s  %10s  %10s\n", "p", "Purity", "Expected")
fmt.Println("-------------------------------")

for i := 0; i <= 15; i++ {
	p := float64(i) * 0.05

	nm := noise.New()
	nm.AddGateError("H", noise.Depolarizing1Q(p))

	sim := densitymatrix.New(1)
	sim.WithNoise(nm)
	if err := sim.Evolve(cH); err != nil {
		panic(err)
	}

	// Depolarizing channel: rho -> (1-p)*rho + (p/3)(X*rho*X + Y*rho*Y + Z*rho*Z)
	// For a pure input, purity = (1 - 4p/3 + 16p^2/9) ... decreases to 0.5 at p=0.75
	expected := 0.5 + 0.5*math.Pow(1-4.0*p/3.0, 2)

	fmt.Printf("%6.2f  %10.6f  %10.6f\n", p, sim.Purity(), expected)
}

fmt.Println("\nPrediction confirmed: purity decreases smoothly from 1.0 to 0.5.")
fmt.Println("The 'Expected' column uses the analytical formula for depolarizing purity.")

## Exercises

### Exercise 1: Model T1 Decay on a Single Qubit

Prepare the $|1\rangle$ state (apply X to $|0\rangle$), then sweep the gate
duration from 0 to $2 \times T_1$ using `ThermalRelaxation`. The $|1\rangle$
population should decay exponentially as $e^{-t/T_1}$. Verify this by
comparing the fidelity to $|1\rangle$ at each time step.

In [ ]:
%%
// Exercise 1: T1 decay of |1> state
// Prepare |1> = X|0>
cX, err := builder.New("x", 1).X(0).Build()
if err != nil {
	panic(err)
}

// Ideal |1> statevector for fidelity
svOne := statevector.New(1)
if err := svOne.Evolve(cX); err != nil {
	panic(err)
}
idealOne := svOne.StateVector()

t1Ex := 100.0 // T1 in ns
t2Ex := 100.0 // T2 = T1 (minimal pure dephasing)

fmt.Printf("%10s  %10s  %10s  %10s\n", "Time (ns)", "Fidelity", "exp(-t/T1)", "Match")
fmt.Println("------------------------------------------------")

for _, gt := range []float64{0, 10, 20, 40, 60, 80, 100, 150, 200} {
	ch := noise.ThermalRelaxation(t1Ex, t2Ex, gt)
	nm := noise.New()
	nm.AddGateError("X", ch)

	sim := densitymatrix.New(1)
	sim.WithNoise(nm)
	if err := sim.Evolve(cX); err != nil {
		panic(err)
	}

	fid := sim.Fidelity(idealOne)
	expDecay := math.Exp(-gt / t1Ex)
	match := math.Abs(fid-expDecay) < 0.02

	fmt.Printf("%10.0f  %10.6f  %10.6f  %10v\n", gt, fid, expDecay, match)
}

fmt.Println("\nFidelity to |1> closely tracks the exponential T1 decay.")

### Exercise 2: Build a Custom Noise Channel

Create a custom **dephasing + bit-flip** channel that applies:
- X error with probability 0.05
- Z error with probability 0.03
- No error with probability 0.92

Use `noise.MustCustom` and apply it to an H gate, then compare the result
with the built-in `BitFlip(0.05)` alone.

In [ ]:
%%
// Exercise 2: Custom combined bit-flip + phase-flip channel
// P(no error) = 0.92, P(X) = 0.05, P(Z) = 0.03
sI := math.Sqrt(0.92)
sX := math.Sqrt(0.05)
sZ := math.Sqrt(0.03)

customBPCh := noise.MustCustom("bitflip_phaseflip", 1, [][]complex128{
	// sqrt(0.92) * I
	{complex(sI, 0), 0, 0, complex(sI, 0)},
	// sqrt(0.05) * X
	{0, complex(sX, 0), complex(sX, 0), 0},
	// sqrt(0.03) * Z
	{complex(sZ, 0), 0, 0, complex(-sZ, 0)},
})

// Apply custom channel
nmBP := noise.New()
nmBP.AddGateError("H", customBPCh)
dmBP := densitymatrix.New(1)
dmBP.WithNoise(nmBP)
if err := dmBP.Evolve(cH); err != nil {
	panic(err)
}

// Compare with just BitFlip(0.05)
nmBF := noise.New()
nmBF.AddGateError("H", noise.BitFlip(0.05))
dmBF := densitymatrix.New(1)
dmBF.WithNoise(nmBF)
if err := dmBF.Evolve(cH); err != nil {
	panic(err)
}

fmt.Println("Custom channel (X=0.05, Z=0.03, I=0.92):")
fmt.Printf("  Purity:   %.6f\n", dmBP.Purity())
fmt.Printf("  Fidelity: %.6f\n", dmBP.Fidelity(idealPlus))

fmt.Println("\nBitFlip(0.05) only:")
fmt.Printf("  Purity:   %.6f\n", dmBF.Purity())
fmt.Printf("  Fidelity: %.6f\n", dmBF.Fidelity(idealPlus))

fmt.Println("\nThe custom channel has lower fidelity because it adds Z errors on top of X errors.")
fmt.Println("This demonstrates how multiple error sources compound.")

## Key Takeaways

1. **Density matrices** generalize statevectors to describe mixed (noisy)
   states. A pure state has purity $\text{Tr}(\rho^2) = 1$; noise reduces
   purity below 1.

2. **Noise channels** are CPTP maps described by Kraus operators
   $\{E_k\}$ satisfying $\sum_k E_k^\dagger E_k = I$. They model the
   interaction between a qubit and its environment.

3. **Depolarizing noise** is the simplest error model (isotropic random
   Pauli errors). **Amplitude damping** models T1 energy relaxation.
   **Phase damping** models pure T2 dephasing. **Bit flip** and **phase flip**
   model single Pauli errors.

4. **T1 and T2 times** are the key hardware parameters. T1 limits how long
   an excited qubit survives. T2 limits how long superpositions remain
   coherent. Longer gate times mean more decoherence.

5. **Fidelity** $F = \langle\psi|\rho|\psi\rangle$ measures how close a noisy
   state is to the ideal. It degrades smoothly with noise strength.

6. **Custom channels** let you model arbitrary error processes from hardware
   characterization data, as long as the Kraus operators are trace-preserving.

7. **Quantum error correction cannot clone** -- the no-cloning theorem
   forces fundamentally different strategies from classical EC. This is
   why quantum error correction requires encoding logical qubits into
   entangled states of many physical qubits.